# Graphagate Training & Evaluation

This notebook trains and tests the **Temporal Graph Network** (TGN) model
locally, without Docker. It uses the available accelerator: **MPS** on Apple
Silicon, **CUDA** on NVIDIA GPUs, otherwise CPU (the Device Detection cell
reports it). On Apple Silicon it applies a temporary monkey-patch of
`tgn.scatter` for a known PyTorch MPS bug with `int64` on
`scatter_reduce_`; on CUDA the patch is inert.


In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import sys

if torch.backends.mps.is_available():
    print("MPS accelerator (Metal) found. The Mac GPU will be used.")
elif torch.cuda.is_available():
    print("CUDA GPU found.")
else:
    print("No GPU found, the CPU will be used.")


## 1. Hyperparameter Configuration
Here we modify the parameters to reflect the new scaling decided in the plan
(e.g. 1000 users + 1000 guests, 2000 devices).


In [ ]:
from graphagate.config import TGNConfig
from graphagate.train_tgn import train_tgn

cfg = TGNConfig(
    # Scale defaults up as discussed
    num_users=1000,
    num_devices=2000,
    num_sources=1500,
    num_configs=400,
    num_events=50000,   # Increase events since we have more entities
    
    # Training paramsx
    epochs=20,           # Keep low for initial testing
    batch_size=256,
    eval_batch_size=128,
    
    # Ensure memory capacity accounts for the extra 1000 guests
    capacity_headroom=2000
)

## 2. Training and Evaluation
We run the full process. The pipeline includes the generation of the updated
synthetic data (with circadian rhythms and lateral movement chains).


In [ ]:
# Temporary patch for the PyTorch MPS bug with int64 on scatter_reduce_
import torch
import torch_geometric.nn.models.tgn as tgn

_original_scatter = tgn.scatter
def _patched_scatter(src, index, dim=0, dim_size=None, reduce='sum'):
    if reduce in ['max', 'amax'] and src.dtype == torch.int64 and getattr(src.device, 'type', '') == 'mps':
        return _original_scatter(src.float(), index, dim, 
                                  dim_size, reduce).long()
    return _original_scatter(src, index, dim, dim_size, reduce)
tgn.scatter = _patched_scatter

# Run the training. The artifacts will be saved in public/
metrics = train_tgn(cfg)

print("\n--- Training Completed ---")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")
